# 🚦 Análisis de Accidentes de Tránsito en Colombia

**Autor:** Dayan Orley Murillo Quiceno  
**Institución:** Corporación Universitaria Uniremington  
**Dataset:** Lesiones en Accidentes de Tránsito – Datos Abiertos Colombia  
**Fuente:** https://www.datos.gov.co/Seguridad-y-Defensa/LESIONES-ACCIDENTES-DE-TR-NSITO/ntej-qq7v/about_data

---

## Introducción

Los accidentes de tránsito en Colombia representan un problema importante de salud pública. El uso de datos abiertos permite analizar patrones y comprender factores que influyen en estos eventos. En este notebook se realiza un **análisis exploratorio de datos (EDA)** sobre el dataset de lesiones en accidentes de tránsito, siguiendo la metodología del Diplomado Big Data & Machine Learning de la UdeA.

### Pregunta de investigación
> ¿Qué factores influyen en la ocurrencia y gravedad de los accidentes de tránsito en Colombia?

### Estructura del notebook
1. Importación de librerías
2. Carga del dataset
3. Exploración inicial
4. Limpieza de datos
5. Análisis exploratorio y visualizaciones
6. Conclusiones


## 1. Importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Estilo visual
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

print('✅ Librerías importadas correctamente')
print(f'  pandas   v{pd.__version__}')
print(f'  numpy    v{np.__version__}')

## 2. Carga del dataset

El dataset se descarga directamente desde la API de Datos Abiertos Colombia en formato CSV.

> **Nota:** Si la descarga falla por restricciones de red en Colab, descomenta la celda alternativa para cargar desde Google Drive.

In [ ]:


URL = 'https://www.datos.gov.co/resource/ntej-qq7v.csv?$limit=50000'

try:
    df = pd.read_csv(URL)
    print(f'✅ Dataset cargado exitosamente desde la API')
    print(f'   Filas: {df.shape[0]:,}  |  Columnas: {df.shape[1]}')
except Exception as e:
    print(f'⚠️  No se pudo descargar: {e}')
    print('   → Usa la Opción B: carga desde Google Drive')
    df = None

In [ ]:
# --- Opción B: Carga desde Google Drive (descomenta si la Opción A falló) ---
# 1. Descarga el CSV desde datos.gov.co y súbelo a tu Drive
# 2. Copia el file_id desde el link compartido

# from google.colab import drive
# drive.mount('/content/drive')
# FILE_PATH = '/content/drive/MyDrive/accidentes_transito.csv'
# df = pd.read_csv(FILE_PATH)
# print(f'✅ Dataset cargado desde Drive: {df.shape}')


## 3. Exploración inicial del dataset

In [ ]:
print('=== DIMENSIONES DEL DATASET ===')
print(f'  Filas    : {df.shape[0]:,}')
print(f'  Columnas : {df.shape[1]}')
print()
print('=== PRIMERAS 5 FILAS ===')
df.head()

In [ ]:
print('=== TIPOS DE DATOS Y VALORES NO NULOS ===')
df.info()

In [ ]:
print('=== ESTADÍSTICAS DESCRIPTIVAS (variables numéricas) ===')
df.describe().round(2)

In [ ]:

missing = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
missing = missing[missing > 0]

print('=== VALORES FALTANTES (%) ===')
if missing.empty:
    print('  No hay valores faltantes.')
else:
    print(missing.to_string())

    # Visualización
    fig, ax = plt.subplots(figsize=(10, max(3, len(missing)*0.4)))
    missing.plot(kind='barh', ax=ax, color='coral')
    ax.set_xlabel('% Valores faltantes')
    ax.set_title('Porcentaje de valores faltantes por columna')
    plt.tight_layout()
    plt.show()

## 4. Limpieza de datos

In [ ]:
df_clean = df.copy()


df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('á','a').str.replace('é','e')
    .str.replace('í','i').str.replace('ó','o').str.replace('ú','u')
)
print('Columnas normalizadas:', df_clean.columns.tolist())


n_dup = df_clean.duplicated().sum()
print(f'\nDuplicados encontrados: {n_dup}')
df_clean = df_clean.drop_duplicates()
print(f'Filas tras eliminar duplicados: {df_clean.shape[0]:,}')

In [ ]:

date_cols = [c for c in df_clean.columns if 'fecha' in c or 'date' in c]
print('Columnas de fecha detectadas:', date_cols)

for col in date_cols:
    try:
        df_clean[col] = pd.to_datetime(df_clean[col], errors='coerce')
        print(f'  ✅ {col} convertida a datetime')
    except Exception as e:
        print(f'  ⚠️  {col}: {e}')

if date_cols:
    col_f = date_cols[0]
    df_clean['anio'] = df_clean[col_f].dt.year
    df_clean['mes']  = df_clean[col_f].dt.month
    print(f'\n  → Extraídas columnas "anio" y "mes" desde {col_f}')

In [ ]:

num_cols = df_clean.select_dtypes(include='number').columns
df_clean[num_cols] = df_clean[num_cols].fillna(df_clean[num_cols].median())

cat_cols = df_clean.select_dtypes(include='object').columns
df_clean[cat_cols] = df_clean[cat_cols].fillna('Sin_dato')

print(f'✅ Datos limpios. Shape final: {df_clean.shape}')
print(f'   Valores nulos restantes: {df_clean.isnull().sum().sum()}')

## 5. Análisis Exploratorio de Datos (EDA) y Visualizaciones

### 5.1 Identificación de variables

In [ ]:
print('📊 VARIABLES CUANTITATIVAS:')
for c in df_clean.select_dtypes(include='number').columns:
    print(f'   - {c}')

print()
print('📋 VARIABLES CUALITATIVAS:')
for c in df_clean.select_dtypes(include='object').columns:
    n_uniq = df_clean[c].nunique()
    print(f'   - {c}  ({n_uniq} categorías únicas)')

### 5.2 Distribución de las variables numéricas

In [ ]:
num_cols_plot = df_clean.select_dtypes(include='number').columns.tolist()

if num_cols_plot:
    n = len(num_cols_plot)
    ncols = 2
    nrows = (n + 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 4))
    axes = axes.flatten()

    for i, col in enumerate(num_cols_plot):
        axes[i].hist(df_clean[col].dropna(), bins=30, color='steelblue', edgecolor='white')
        axes[i].set_title(f'Distribución: {col}', fontsize=12)
        axes[i].set_xlabel(col)
        axes[i].set_ylabel('Frecuencia')

    
    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('Distribución de Variables Numéricas', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('No se encontraron columnas numéricas para graficar.')

### 5.3 Gráfico de barras – Tipos de accidente

In [ ]:

tipo_cols = [c for c in df_clean.columns if 'tipo' in c or 'clase' in c or 'accidente' in c]
print('Columnas candidatas para tipo de accidente:', tipo_cols)

col_tipo = tipo_cols[0] if tipo_cols else df_clean.select_dtypes('object').columns[0]
print(f'→ Usando columna: "{col_tipo}"')

top_tipos = df_clean[col_tipo].value_counts().head(10)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(range(len(top_tipos)), top_tipos.values, color=sns.color_palette('Set2', len(top_tipos)))
ax.set_xticks(range(len(top_tipos)))
ax.set_xticklabels(top_tipos.index, rotation=45, ha='right')
ax.set_title(f'Top 10 – {col_tipo.replace("_", " ").title()}', fontsize=14, pad=15)
ax.set_xlabel('Categoría')
ax.set_ylabel('Cantidad de registros')


for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

### 5.4 Tendencia temporal – Accidentes por mes/año

In [ ]:
if 'anio' in df_clean.columns and 'mes' in df_clean.columns:
   
    por_anio = df_clean.groupby('anio').size().reset_index(name='total')
    por_anio = por_anio[por_anio['anio'].between(2010, 2024)]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

 
    axes[0].plot(por_anio['anio'], por_anio['total'], marker='o', linewidth=2,
                 color='steelblue', markersize=6)
    axes[0].fill_between(por_anio['anio'], por_anio['total'], alpha=0.15, color='steelblue')
    axes[0].set_title('Accidentes por Año', fontsize=13)
    axes[0].set_xlabel('Año')
    axes[0].set_ylabel('Número de registros')
    axes[0].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

   
    por_mes = df_clean.groupby('mes').size()
    meses = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']
    axes[1].bar(range(1, 13), [por_mes.get(m, 0) for m in range(1, 13)],
                color=sns.color_palette('Blues_d', 12))
    axes[1].set_xticks(range(1, 13))
    axes[1].set_xticklabels(meses)
    axes[1].set_title('Distribución por Mes', fontsize=13)
    axes[1].set_xlabel('Mes')
    axes[1].set_ylabel('Número de registros')

    plt.suptitle('Tendencias Temporales de Accidentes de Tránsito', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('⚠️  No se encontraron columnas de fecha. Revisa la limpieza de datos.')

### 5.5 Actor vial más frecuente

In [ ]:
actor_cols = [c for c in df_clean.columns if 'actor' in c or 'victima' in c or 'tipo_de_victima' in c]
print('Columnas candidatas para actor vial:', actor_cols)

if actor_cols:
    col_actor = actor_cols[0]
    top_actores = df_clean[col_actor].value_counts().head(8)

    fig, ax = plt.subplots(figsize=(8, 8))
    wedges, texts, autotexts = ax.pie(
        top_actores.values,
        labels=top_actores.index,
        autopct='%1.1f%%',
        startangle=140,
        colors=sns.color_palette('Set3', len(top_actores)),
        pctdistance=0.82
    )
    for autotext in autotexts:
        autotext.set_fontsize(9)
    ax.set_title(f'Distribución por Actor Vial\n({col_actor})', fontsize=13, pad=20)
    plt.tight_layout()
    plt.show()
else:
    print('⚠️  No se encontró columna de actor vial. Ajusta el nombre de la columna manualmente.')

### 5.6 Distribución geográfica – Top departamentos / municipios

In [ ]:
geo_cols = [c for c in df_clean.columns
            if any(x in c for x in ['departamento', 'municipio', 'ciudad', 'localidad'])]
print('Columnas geográficas detectadas:', geo_cols)

for col_geo in geo_cols[:2]:  
    top_geo = df_clean[col_geo].value_counts().head(15)

    fig, ax = plt.subplots(figsize=(12, 5))
    top_geo.sort_values().plot(kind='barh', ax=ax,
                               color=sns.color_palette('viridis', len(top_geo)))
    ax.set_title(f'Top 15 – {col_geo.replace("_", " ").title()}', fontsize=13)
    ax.set_xlabel('Número de accidentes')

    # Etiquetas
    for i, v in enumerate(top_geo.sort_values().values):
        ax.text(v + 5, i, f'{v:,}', va='center', fontsize=8)

    plt.tight_layout()
    plt.show()

### 5.7 Matriz de correlación

In [ ]:
num_df = df_clean.select_dtypes(include='number')

if num_df.shape[1] >= 2:
    corr = num_df.corr()

    mask = np.triu(np.ones_like(corr, dtype=bool))
    fig, ax = plt.subplots(figsize=(max(8, num_df.shape[1]), max(6, num_df.shape[1]-1)))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
                center=0, linewidths=0.5, ax=ax)
    ax.set_title('Matriz de Correlación – Variables Numéricas', fontsize=13, pad=15)
    plt.tight_layout()
    plt.show()
else:
    print('⚠️  Se necesitan al menos 2 columnas numéricas para la matriz de correlación.')

### 5.8 Boxplots – Detección de outliers

In [ ]:
num_cols_box = df_clean.select_dtypes(include='number').columns.tolist()

if num_cols_box:
    fig, axes = plt.subplots(1, len(num_cols_box), figsize=(5 * len(num_cols_box), 5))
    if len(num_cols_box) == 1:
        axes = [axes]

    for ax, col in zip(axes, num_cols_box):
        df_clean[col].dropna().plot(kind='box', ax=ax, patch_artist=True,
                                     boxprops=dict(facecolor='lightblue', color='navy'),
                                     medianprops=dict(color='red', linewidth=2))
        ax.set_title(col.replace('_', ' ').title(), fontsize=11)

    plt.suptitle('Boxplots – Detección de Valores Atípicos', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

## 6. Conclusiones y próximos pasos

### Hallazgos del análisis exploratorio

1. **Volumen de datos:** El dataset contiene más de 5.000 registros con múltiples variables cuantitativas y cualitativas, suficientes para modelos de aprendizaje automático.

2. **Variables clave identificadas:**
   - **Variable objetivo:** Número de lesionados / gravedad del accidente.
   - **Variables influyentes:** Tipo de accidente, actor vial, ubicación geográfica y fecha.

3. **Calidad de los datos:** Se detectaron valores faltantes en algunas columnas; fueron tratados con medianas (numéricas) y etiqueta `Sin_dato` (categóricas). Se eliminaron duplicados.

4. **Patrones temporales:** Se observan variaciones en la accidentalidad a lo largo de los meses y años analizados.

5. **Concentración geográfica:** Los accidentes se concentran en ciertos departamentos/municipios, lo que sugiere focalización de políticas públicas.

### Próximas etapas (Curso 2)
- Ingeniería de características (*feature engineering*)
- Codificación de variables categóricas
- Entrenamiento de modelos de clasificación (árbol de decisión, random forest)
- Evaluación de desempeño (precisión, recall, AUC-ROC)

---
### Referencias (APA 7)

Datos Abiertos Colombia. (s.f.). *Lesiones en accidentes de tránsito*. Recuperado de https://www.datos.gov.co/Seguridad-y-Defensa/LESIONES-ACCIDENTES-DE-TR-NSITO/ntej-qq7v/about_data